In [2]:
import numpy as np
import scipy.special as special

class CuckooSearchOptimizer:
    def __init__(self, objective_function, lower_bound, upper_bound,
                 dimension, num_nests=25, discovery_rate=0.25, max_iterations=500):
        self.obj_func = objective_function
        self.lb = np.ones(dimension) * lower_bound
        self.ub = np.ones(dimension) * upper_bound
        self.dim = dimension
        self.n = num_nests
        self.pa = discovery_rate
        self.max_iter = max_iterations

        self.beta = 1.5
        self._calculate_mantegna_sigma()

        self.nests = np.zeros((self.n, self.dim))
        self.fitness = np.zeros(self.n)
        self.best_nest = None
        self.best_fitness = float('inf')

    def _calculate_mantegna_sigma(self):
        num = special.gamma(1 + self.beta) * np.sin(np.pi * self.beta / 2)
        den = special.gamma((1 + self.beta) / 2) * self.beta * (2 ** ((self.beta - 1) / 2))
        self.sigma = (num / den) ** (1 / self.beta)

    def initialize_population(self):
        self.nests = self.lb + (self.ub - self.lb) * np.random.rand(self.n, self.dim)
        for i in range(self.n):
            self.fitness[i] = self.obj_func(self.nests[i])

        best_idx = np.argmin(self.fitness)
        self.best_fitness = self.fitness[best_idx]
        self.best_nest = np.copy(self.nests[best_idx])

    def _levy_flight_step(self, nest_position):
        u = np.random.normal(0, self.sigma, size=self.dim)
        v = np.random.normal(0, 1, size=self.dim)
        step = u / (np.abs(v) ** (1 / self.beta))

        step_size = 0.01 * step * (nest_position - self.best_nest)
        new_position = nest_position + step_size * np.random.normal(size=self.dim)
        return new_position

    def _clip_bounds(self, position):
        return np.clip(position, self.lb, self.ub)

    def optimize(self):
        self.initialize_population()

        for iteration in range(self.max_iter):
            i = np.random.randint(0, self.n)
            new_position = self._levy_flight_step(self.nests[i])
            new_position = self._clip_bounds(new_position)
            new_fitness = self.obj_func(new_position)

            j = np.random.randint(0, self.n)
            if new_fitness < self.fitness[j]:
                self.nests[j] = new_position
                self.fitness[j] = new_fitness

            discovery_mask = np.random.rand(self.n) < self.pa

            perm1 = np.random.permutation(self.n)
            perm2 = np.random.permutation(self.n)

            step_size = np.random.rand(self.n, self.dim) * (self.nests[perm1] - self.nests[perm2])

            for k in range(self.n):
                if discovery_mask[k]:
                    new_nest = self.nests[k] + step_size[k]
                    self.nests[k] = self._clip_bounds(new_nest)
                    self.fitness[k] = self.obj_func(self.nests[k])

            current_best_idx = np.argmin(self.fitness)
            if self.fitness[current_best_idx] < self.best_fitness:
                self.best_fitness = self.fitness[current_best_idx]
                self.best_nest = np.copy(self.nests[current_best_idx])

        return self.best_nest, self.best_fitness

if __name__ == "__main__":
    def rastrigin_function(x):
        A = 10
        return A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x))

    try:
        problem_dimensions = int(input("Enter problem dimensions: "))
        lower_boundary = float(input("Enter lower boundary: "))
        upper_boundary = float(input("Enter upper boundary: "))
    except ValueError:
        print("Invalid input. Using defaults: dim=5, lb=-5.12, ub=5.12")
        problem_dimensions = 5
        lower_boundary = -5.12
        upper_boundary = 5.12

    optimizer = CuckooSearchOptimizer(
        objective_function=rastrigin_function,
        lower_bound=lower_boundary,
        upper_bound=upper_boundary,
        dimension=problem_dimensions,
        num_nests=25,
        discovery_rate=0.25,
        max_iterations=1000
    )

    best_solution, optimal_value = optimizer.optimize()

    print("--- Optimization Completed ---")
    print(f"Optimal Coordinates Found:\n {best_solution}")
    print(f"Minimum Value Reached: {optimal_value:.6f}")

Enter problem dimensions: 4
Enter lower boundary: 10
Enter upper boundary: 17
--- Optimization Completed ---
Optimal Coordinates Found:
 [10. 10. 10. 10.]
Minimum Value Reached: 400.000000
